In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import matplotlib.pyplot as plt
import numpy as np

# a. Set up environment & b. Loading and preprocessing data
IMG_SIZE = 224
BATCH_SIZE = 32

# Load Fashion MNIST from Keras built-in datasets
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
NUM_CLASSES = len(class_names)

# Convert to tf.data.Dataset
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

train_ds = train_ds.shuffle(10000).batch(BATCH_SIZE)
test_ds = test_ds.batch(BATCH_SIZE)

# Preprocessing function: resize to 224x224, convert grayscale (1 channel) to RGB (3 channels), and apply ResNet preprocessing
def preprocess(images, labels):
    images = tf.expand_dims(images, axis=-1)
    images = tf.image.grayscale_to_rgb(images)
    images = tf.image.resize(images, (IMG_SIZE, IMG_SIZE))
    images = tf.keras.applications.resnet50.preprocess_input(images)
    labels = tf.one_hot(labels, NUM_CLASSES)
    return images, labels

# For visualization later, we keep a raw version of test images (resized to 224x224)
def preprocess_viz(images, labels):
    images = tf.expand_dims(images, axis=-1)
    images = tf.image.grayscale_to_rgb(images)
    images = tf.image.resize(images, (IMG_SIZE, IMG_SIZE))
    return images, labels

viz_ds = test_ds.map(preprocess_viz)

train_ds = train_ds.map(preprocess)
test_ds = test_ds.map(preprocess)

# c. Choose the pre-trained Model (Transfer Learning)
base_model = tf.keras.applications.ResNet50(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# d. Freeze the base layers and add custom layers for fine tuning
base_model.trainable = False

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# e. Object detection / classification training & f. Compute loss
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("--- Training Initial Custom Head ---")
history = model.fit(train_ds, epochs=3, validation_data=test_ds)

# g. Evaluate the Model
loss, accuracy = model.evaluate(test_ds)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# h. Visualize the prediction
for raw_images, labels in viz_ds.take(1):
    processed_input = tf.keras.applications.resnet50.preprocess_input(raw_images)
    preds = model.predict(processed_input)
    
    pred_idx = np.argmax(preds[0])
    true_idx = labels[0].numpy()
    
    plt.figure(figsize=(6, 6))
    plt.imshow(raw_images[0].numpy().astype("uint8"))
    plt.title(f"True: {class_names[true_idx]} | Pred: {class_names[pred_idx]}")
    plt.axis('off')
    plt.show()
    break

# i. Fine Tune the Model (Unfreeze top layers of base model)
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5), # Lower learning rate for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("--- Fine-Tuning Model ---")
history_fine = model.fit(train_ds, epochs=3, validation_data=test_ds)